# Microzone Cleanup Notebook

This notebook takes a microzone and traffic analysis zone layer and fills in gaps, then merges them into neighbors without crossing over the taz boundry.

Point workspace variable at the `.gdb` folder for your project

Ensure that microzone and traffic analysis zone are inside of your `.gdb` folder

Point `maz` variable at the name of microzone feature class 

Example: `maz = "microzone_layer"`

Point `taz` variable at the name of traffic analysis zone feature class

Example: `taz = "taz_layer"`




### Import Necessary Libraries

In [1]:
import arcpy

### Assign key Variables

In [2]:
workspace = r"D:\ABM-INP-Microzones\Scripts\Scratch\Microzones_Cleaner\Microzones_Cleaner.gdb"
maz = "microzones_Export"
taz = "TAZ_Export"

### Point Arcpy at `.gdb` folder
Enable output overwrite

In [3]:
arcpy.env.workspace = workspace
arcpy.env.overwriteOutput = True

### Remove MAG TAZ
Since the TAZ layer inclues MAG TAZ's we can exclude them from our file since we only want to look at WFRC TAZ

In [4]:
temp_taz = "temp_taz"
wfrc_taz = "_01_wfrc_taz"


arcpy.management.MakeFeatureLayer(
    in_features=taz, 
    out_layer=temp_taz)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=temp_taz, 
    selection_type="NEW_SELECTION", 
    where_clause="PLANAREA = 'WFRC' ")

arcpy.management.CopyFeatures(
    in_features=temp_taz,
    out_feature_class=wfrc_taz)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_01_wfrc_taz'>

### Run Identity Tool

This will fill in all gaps between polygons

Creates a new feature class called `maz_i` which should have more features than orignial maz becuase these are all new filled in featuers

Original Microzones total = 9,875

MAZID"0" = 861

New Zones = 869

MAZ identified  total = 10,744

In [5]:
maz_i = "_02_maz_i"

arcpy.analysis.Identity(
    in_features=wfrc_taz, 
    identity_features=maz, 
    out_feature_class=maz_i)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_02_maz_i'>

### Multipart to Singlepart

Identity will treat some new features a single one and this function will break them back into their own

Creates a new feature class called `maz_filled` with all gaps filled in as unique features

MAZ Filled total = 11,680

New Zones = 936

MAZID"0" = 1,793

NEW MAZID"0" = 924

Missing duplicate Zones = 12
The 12 duplicate zones will be filterd out later when I look for sliver zones


In [6]:
maz_filled = "_03_maz_filled"
arcpy.management.MultipartToSinglepart(
    in_features= maz_i,
    out_feature_class=maz_filled)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_03_maz_filled'>

### Create new field for filled gap polygon

All new polygons created from the identify and multipart functions will have `MAZID = 0` so to easily identify them later I will make a dummy column for it
Flags 1,793 polygons


In [7]:
arcpy.management.CalculateField(
    in_table=maz_filled,
    field="gap_flag",
    expression="1 if (!MAZID! == 0 or !MAZID! is None) else 0",
    expression_type="PYTHON3",
    field_type="SHORT"
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_03_maz_filled'>

## Identify zones for review

In addition to gap zones we want to flag sliver zones

Shape Area is less than 10,000

AND 

PolsbyPopper is less than .03

OR 

PerimtertoArea is less than .03

OR

Shape Area < 1,000


**This will flag 1,986 zones as sliver zones to be merged into the neighboring microzones**

In [8]:
maz_sliver = "_04_maz_sliver"


arcpy.management.CopyFeatures(
    in_features= maz_filled,
    out_feature_class=maz_sliver
)

code_block = """
def check_sliver(pp, pa, area, gap):
    # Guard clause: large features (highways) are never slivers
    if area is not None and area >= 10000:
        return 0
    
    # Check sliver criteria for features under 10,000
    if pp is not None and pp < 0.03:
        return 1
    if pa is not None and pa < 0.03:
        return 1
    if area is not None and area < 1000:
        return 1
    if gap == 1:
        return 1
    return 0
"""

arcpy.management.CalculateField(
    in_table=maz_sliver,
    field="Sliver_Flag",
    expression="check_sliver(!PolsbyPopper!, !PerimtertoArea!, !Shape_Area!, !gap_flag!)",
    expression_type="PYTHON3",
    code_block=code_block,
    field_type="SHORT"
)


<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_04_maz_sliver'>

### Additional layer of just sliver zones for eye test

In [9]:
slivers = "_05_iso_slivers"

arcpy.management.MakeFeatureLayer(
    in_features=maz_sliver, 
    out_layer=slivers
)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=slivers, 
    selection_type="NEW_SELECTION", 
    where_clause="Sliver_Flag = 1"
)

arcpy.management.CopyFeatures(
    in_features=slivers,
    out_feature_class=slivers
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_05_iso_slivers'>

### Create Taz vector lines

Eliminate tool requires vector lines to use as a boarder so we dont merge between TAZ

In [10]:
taz_lines = "_06_taz_lines"
arcpy.management.PolygonToLine(taz, taz_lines)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_06_taz_lines'>

### Run Eliminate 

Merges 1,810 features which leaves 176 unmerged features

These are zones that have micro crossover TAZ boundaries



In [11]:
maz_cleaned = "_07_maz_cleaned"
elim_layer = "temp_elim_layer"

arcpy.management.MakeFeatureLayer(
    in_features=maz_sliver, 
    out_layer=elim_layer
)

arcpy.management.SelectLayerByAttribute(elim_layer, "NEW_SELECTION", 'Sliver_Flag = 1')

arcpy.management.Eliminate(
    in_features=elim_layer, 
    out_feature_class=maz_cleaned,
    ex_features=taz_lines
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_07_maz_cleaned'>

### Snap micro zones to the taz lines to get rid of micro cross over

These zones are all small enough to be automatically deleted with Integrate tool

In [12]:
maz_snapped = "_08_maz_snapped"

arcpy.management.CopyFeatures(maz_cleaned, maz_snapped)
arcpy.management.Integrate(
    in_features=[[maz_snapped, 1]], 
    cluster_tolerance="1 Meters" # Adjust to "3 Feet" if your projection is in feet
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_08_maz_snapped'>

### Check for microzones that overlap TAZ

2,093 microzones that cross over taz boundaries

In [13]:
snapped_layer = "temp_snapped_layer"
maz_crossover = "_09_maz_crossover"

arcpy.management.MakeFeatureLayer(maz_snapped, snapped_layer)

arcpy.management.SelectLayerByLocation(
    in_layer=snapped_layer,
    overlap_type="CROSSED_BY_THE_OUTLINE_OF",
    select_features=taz
)

arcpy.management.CopyFeatures(snapped_layer, maz_crossover)



<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_09_maz_crossover'>

### Visualize overlapping areas

Shrink all maz by .01 meters to ignore zones that are perfecty ontop of the taz. 

clip all the new maz to taz boundaries

In [14]:
maz_shrunk = "_10_maz_shrunk"
maz_crossover_lines = "_11_maz_crossover_lines"

arcpy.analysis.Buffer(
    in_features=maz_snapped, 
    out_feature_class=maz_shrunk, 
    buffer_distance_or_field="-0.01 Meters"
)


arcpy.analysis.Clip(
    in_features=taz_lines,
    clip_features=maz_shrunk,
    out_feature_class=maz_crossover_lines
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_11_maz_crossover_lines'>

### Split crossover microzones at taz boundaries
### Merge the new cuts with neighboring MAZ
Use TAZ lines to make sure it doesn't merge back over TAZ boundaries

In [32]:
maz_taz_identity = "_12_maz_taz_identity"
maz_cross_2 = "_13_maz_crossing"
temp_elim = "temp_elim"

arcpy.analysis.Identity(
    in_features=maz_snapped,
    identity_features=taz,
    out_feature_class=maz_taz_identity
)

arcpy.management.MakeFeatureLayer(maz_taz_identity, temp_elim)
# 2. Flag the tiny crossover slivers (where the Microzone's parent TAZ doesn't match the physical TAZ it spilled into)
# Assuming your MAZ has a 'TAZ_ID' field and the TAZ layer has a 'TAZ_ID' field

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=temp_elim,
    selection_type="NEW_SELECTION",
    where_clause="TAZID_1 <> TAZID_12"
)

selected_count = int(arcpy.management.GetCount(temp_elim)[0])
print(f"Selected {selected_count} crossover slivers for elimination.")

# 3. Only run Eliminate if features were successfully selected
if selected_count > 0:
    arcpy.management.Eliminate(
        in_features=temp_elim,
        out_feature_class=maz_cross_2,
        ex_features=taz_lines # This correctly prevents them from merging back across the TAZ line
    )
else:
    print("Eliminate skipped: No features selected. Double check your field names.")

Selected 14396 crossover slivers for elimination.


Identiy created 14,396 new sliver zones where microzones cross over a taz line

but the eliminate tool wasn't able to merge all of them into new zones. 

There are 1,287 polygons that didn't merge because its either on the outer edgeof the map or in an island of slivers

### Create new layer of just these new sliver zones

In [33]:
stuck_slivers = "_14_stuck_slivers"
temp_stuck = "temp_stuck_layer"

# Create a layer from your Eliminate output
arcpy.management.MakeFeatureLayer(
    in_features=maz_cross_2, 
    out_layer=temp_stuck
)

# Use the exact same WHERE clause from your Eliminate step
# (Make sure to use the correct suffix, e.g., TAZID_1 or TAZID_2)
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=temp_stuck,
    selection_type="NEW_SELECTION",
    where_clause="TAZID_1 <> TAZID_12" 
)

# Export them to their own layer so you can inspect them
arcpy.management.CopyFeatures(
    in_features=temp_stuck, 
    out_feature_class=stuck_slivers
)

# Print the count to confirm
stuck_count = int(arcpy.management.GetCount(temp_stuck)[0])
print(f"Found {stuck_count} orphaned polygons that failed to merge.")

Found 1287 orphaned polygons that failed to merge.


In [34]:
maz_final_cleaned = "_15_maz_final_cleaned"
temp_small_polys = "temp_small_polys"

# 1. Create a feature layer from the output of your previous eliminate step
arcpy.management.MakeFeatureLayer(
    in_features=maz_cross_2, 
    out_layer=temp_small_polys
)

# 2. Select the stuck microzones under 12 square meters
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=temp_small_polys,
    selection_type="NEW_SELECTION",
    where_clause="Shape_Area < 12"
)

# 3. Eliminate them into their neighboring microzones, blocking crossover with taz_lines
arcpy.management.Eliminate(
    in_features=temp_small_polys,
    out_feature_class=maz_final_cleaned,
    ex_features=taz_lines
)

# Optional: Print how many were cleaned up
cleaned_count = int(arcpy.management.GetCount(temp_small_polys)[0])
print(f"Eliminated {cleaned_count} small polygons.")

Eliminated 1287 small polygons.


### Rerun Crossover Check

In [36]:
final_check_layer = "temp_final_check"
maz_final_crossover = "_16_maz_final_crossover"

# 1. Create a temporary layer from your cleaned microzones
arcpy.management.MakeFeatureLayer(
    in_features=maz_final_cleaned, 
    out_layer=final_check_layer
)

# 2. Select any microzones that still cross the TAZ boundaries
arcpy.management.SelectLayerByLocation(
    in_layer=final_check_layer,
    overlap_type="CROSSED_BY_THE_OUTLINE_OF",
    select_features=taz
)

# 3. Export the selection to a new feature class so you can view it
arcpy.management.CopyFeatures(
    in_features=final_check_layer, 
    out_feature_class=maz_final_crossover
)

# 4. Print the result
final_crossover_count = int(arcpy.management.GetCount(final_check_layer)[0])
if final_crossover_count == 0:
    print("Success! 0 crossover zones found. The boundaries are clean.")
else:
    print(f"Check complete: Found {final_crossover_count} overlapping zones remaining.")

Check complete: Found 3471 overlapping zones remaining.


### Visualize Overlap

summary stats for maz per taz, county, city 

count

min

max

avg



be able to add in the MAG microzones